# Measuring trend in futures

Scaffold for the signal stage of the pipeline: (1) a catalogue of ways to measure trend, (2) a common interface and a first set of implementations, (3) a per-market test grid, (4) the overfitting checks every result from the grid must pass before it is believed.

Everything here is **exploratory**. Nothing is selected from this notebook without a frozen rule set and untouched data (see TODO: Validation). Data are the cleaned monthly returns used by the TSMOM baseline, 1976-2014; the 2005-2014 window has already been inspected and is not a holdout.

## Summary: which signal for which asset class

| Asset class | Signal used | Evidence from section 5 (in-sample, gross, class-portfolio Sharpe) |
|---|---|---|
| Commodities (COMM) | baseline: 12-month sum (`sum_12`) | 0.83; ranks 6th of 126 variants, best variant is only 0.05 higher |
| Fixed income (FI) | baseline: 12-month sum (`sum_12`) | 0.56; ranks 5th of 126, best variant is only 0.05 higher |
| Equities (EQ) | baseline: 12-month sum (`sum_12`) | 0.55, ranks 1st of 126 variants; the skip-month rule scored 0.51 and cut equities' net contribution in the pipeline from 1.59% to 1.43% a year, so it was dropped |
| FX | blend of `sum_3` and `sum_12` (average of the two signs) | `sum_3` alone scored 0.59 against 0.55 for `sum_12` in-sample but doubled FX turnover in the pipeline; the average is a hedge between the two. This exact blend was not in the notebook's grid |

These are **design choices, not validated results**. They come from in-sample class-level rankings that section 7 has not yet priced, and the same notebook finds that no combination or alternative beats the fixed 12-month sum on the pooled portfolio (sections 6 and 7). Treat the FX row as a hypothesis to confirm with a class-level out-of-sample test. In the pipeline (`qps_v2`, endpoint-sensitivity run, 10% band, 5 bps) these rules gave a Sharpe of 1.07 against 1.06 for the all-baseline portfolio, a difference of 0.3% a year with a t-statistic of 1.1: not distinguishable from zero. The numbers above are from the run saved with this notebook.

## 1. Trend measures

### 1A. The families: what they are, pros and cons, and what TSMOM and QPS use

| Family | Idea | Pros | Cons | TSMOM / QPS |
|---|---|---|---|---|
| Trailing return | Past return over a window predicts the next period's sign | Simplest and most transparent; few parameters; the documented anomaly (Moskowitz-Ooi-Pedersen 2012); works from returns alone | One window choice; equal weight inside the window; slow at turns; whipsaws in ranges | **Both use it**: sign of the sum of the last 12 monthly returns (`signal_months = 12`, `SumReturnSignal`) |
| Moving average | Price relative to its own smoothed history; ARMA/ARIMA learn the weights | Familiar to practitioners; smooth lines; crossover filters noise | Same information as trailing return with different weights, so a large grid adds trials but little new content; lags more; ARMA/ARIMA add order selection, refitting and unstable fits | Not used |
| Range / breakout | New highs or lows relative to a trailing range | Reacts to new extremes; ignores path inside the range; natural stop level | Depends on the window; discards most of the data; unstable at range edges; better with daily bars | Not used |
| Statistical | Fit a trend model (slope t-stat, variance ratio, Kalman, HMM) | Gives strength and uncertainty, and can test whether trends exist in a market | More parameters and assumptions; outlier-sensitive; easiest family to overfit | Not used |

**Where the two strategies differ.** TSMOM and QPS use the *same* trend measure. The difference lies downstream of the signal: QPS compares trend against an always-long STATIC book and adds correlation-aware risk, a 2% volatility floor, a 3x gross cap, a 10% forecast risk ceiling, a +/-10% trading band and transaction costs. So this notebook tests whether another trend measure could replace the 12-month sign, holding those stages fixed.

**Why the trailing-return family is the benchmark here.** It is the only family with a published, out-of-sample record on this asset set, and every other implementable measure is a re-weighting of the same price history. A new measure only matters if it beats the 12-month sign after the checks in section 5, not just on a heatmap.

#### Catalogue of measures in these families

"Trend" has three separable parts: **direction** (sign), **strength** (how confident), and **persistence** (does the trend last, i.e. is it worth following). Most measures give direction and strength; only a few address persistence.

Row colour = family: <span style="background:rgba(66,133,244,0.35);padding:1px 6px">Trailing return</span>, <span style="background:rgba(52,168,83,0.35);padding:1px 6px">Moving average</span>, <span style="background:rgba(251,140,0,0.35);padding:1px 6px">Range</span>, <span style="background:rgba(156,39,176,0.35);padding:1px 6px">Statistical</span>.  Notation: r<sub>t</sub> monthly return; p<sub>t</sub> = Σ log(1+r) the log price index; <i>L</i>, <i>N</i> windows in months; σ<sub>t</sub> trailing volatility.

<table>
<tr><th style="padding:4px 8px;text-align:left;vertical-align:top">Family</th><th style="padding:4px 8px;text-align:left;vertical-align:top">Measure</th><th style="padding:4px 8px;text-align:left;vertical-align:top">Definition</th><th style="padding:4px 8px;text-align:left;vertical-align:top">Equation</th><th style="padding:4px 8px;text-align:left;vertical-align:top">Parameters</th><th style="padding:4px 8px;text-align:left;vertical-align:top">Notes</th></tr>
<tr style="background:rgba(66,133,244,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Trailing return</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Sum / compounded return</td><td style="padding:4px 8px;text-align:left;vertical-align:top">sign of the last <i>L</i> monthly returns</td><td style="padding:4px 8px;text-align:left;vertical-align:top">sign( Σ<sub>i=0..L-1</sub> r<sub>t-i</sub> ), or Σ log(1+r<sub>t-i</sub>) when compounded</td><td style="padding:4px 8px;text-align:left;vertical-align:top"><i>L</i> (MOP 2012: 12)</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: the net price change over the last <i>L</i> months. Up means long, down means short. Every month in the window counts equally and the path taken is ignored. Sum and compounded differ only through compounding. This is the TSMOM baseline</td></tr>
<tr style="background:rgba(66,133,244,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Trailing return</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Skip-month</td><td style="padding:4px 8px;text-align:left;vertical-align:top">return over months t-L+1..t-s, excluding the latest <i>s</i> months</td><td style="padding:4px 8px;text-align:left;vertical-align:top">sign( p<sub>t-s</sub> − p<sub>t-L</sub> )</td><td style="padding:4px 8px;text-align:left;vertical-align:top"><i>L</i>, <i>s</i></td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: the same net change, but leaves out the most recent month(s). Tests whether the latest move is noise or a short-term bounce that reverses, as often seen in equity momentum</td></tr>
<tr style="background:rgba(66,133,244,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Trailing return</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Multi-horizon blend</td><td style="padding:4px 8px;text-align:left;vertical-align:top">average of signs over several <i>L</i> (e.g. 1, 3, 12)</td><td style="padding:4px 8px;text-align:left;vertical-align:top">(1/K) Σ<sub>k</sub> sign( p<sub>t</sub> − p<sub>t-L<sub>k</sub></sub> )</td><td style="padding:4px 8px;text-align:left;vertical-align:top">set of <i>L</i>, weights</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: whether short, medium and long windows agree. The score is +1 or -1 when all horizons agree and in between when they disagree, so it reflects how broad the trend is across time scales</td></tr>
<tr style="background:rgba(66,133,244,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Trailing return</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Return / volatility (t-stat)</td><td style="padding:4px 8px;text-align:left;vertical-align:top">trailing mean over its standard error</td><td style="padding:4px 8px;text-align:left;vertical-align:top">tanh( r̄<sub>L</sub> / ( s<sub>L</sub> / √L ) )</td><td style="padding:4px 8px;text-align:left;vertical-align:top"><i>L</i></td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: how large the average return is compared with its noise. A steady +10% gives a strong score; a +10% reached through large swings gives a weak one. Measures the quality of the trend, not just its direction</td></tr>
<tr style="background:rgba(52,168,83,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Moving average</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Price vs MA</td><td style="padding:4px 8px;text-align:left;vertical-align:top">sign of price minus <i>L</i>-month average</td><td style="padding:4px 8px;text-align:left;vertical-align:top">sign( p<sub>t</sub> − (1/L) Σ<sub>i=0..L-1</sub> p<sub>t-i</sub> )</td><td style="padding:4px 8px;text-align:left;vertical-align:top"><i>L</i></td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: whether today's price is above or below its recent average. Above means the market has been rising relative to its history. It uses the same data as trailing return but weights recent prices more heavily in effect</td></tr>
<tr style="background:rgba(52,168,83,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Moving average</td><td style="padding:4px 8px;text-align:left;vertical-align:top">MA crossover</td><td style="padding:4px 8px;text-align:left;vertical-align:top">fast MA vs slow MA</td><td style="padding:4px 8px;text-align:left;vertical-align:top">sign( MA<sub>fast</sub>(p) − MA<sub>slow</sub>(p) )</td><td style="padding:4px 8px;text-align:left;vertical-align:top">fast, slow</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: whether the short-term average is above the long-term average, i.e. whether recent prices are higher than older ones. Smoothing both averages filters noise at the cost of turning late</td></tr>
<tr style="background:rgba(52,168,83,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Moving average</td><td style="padding:4px 8px;text-align:left;vertical-align:top">EWMAC</td><td style="padding:4px 8px;text-align:left;vertical-align:top">fast minus slow exponential MA, divided by volatility</td><td style="padding:4px 8px;text-align:left;vertical-align:top">tanh( ( EMA<sub>fast</sub>(p) − EMA<sub>slow</sub>(p) ) / σ<sub>t</sub> )</td><td style="padding:4px 8px;text-align:left;vertical-align:top">fast span, slow span</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: the gap between a fast and a slow average that weights recent prices most, scaled by volatility. A larger gap means a stronger trend, so positions grow and shrink smoothly (Carver) instead of flipping</td></tr>
<tr style="background:rgba(52,168,83,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Moving average</td><td style="padding:4px 8px;text-align:left;vertical-align:top">ARMA / ARIMA</td><td style="padding:4px 8px;text-align:left;vertical-align:top">fit an ARMA(p,q) to returns (or ARIMA(p,1,q) to log price) and take the sign of the one-step forecast</td><td style="padding:4px 8px;text-align:left;vertical-align:top">r<sub>t</sub> = Σ φ<sub>i</sub> r<sub>t-i</sub> + Σ θ<sub>j</sub> ε<sub>t-j</sub> + ε<sub>t</sub>; signal = sign( r̂<sub>t+1|t</sub> )</td><td style="padding:4px 8px;text-align:left;vertical-align:top">p, d, q; fit window</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: whether past returns and past forecast errors help predict the next return. The weights are estimated from the data rather than fixed, so it learns how much recent history matters. Refit each month on a trailing window only. <b>Not yet in the registry</b></td></tr>
<tr style="background:rgba(251,140,0,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Range</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Breakout (Donchian)</td><td style="padding:4px 8px;text-align:left;vertical-align:top">position of price inside its trailing <i>N</i>-month high/low range</td><td style="padding:4px 8px;text-align:left;vertical-align:top">2 ( p<sub>t</sub> − min<sub>N</sub> p ) / ( max<sub>N</sub> p − min<sub>N</sub> p ) − 1</td><td style="padding:4px 8px;text-align:left;vertical-align:top"><i>N</i></td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: where price sits between its recent low and high. Near the top means a new high is being set (uptrend), near the bottom a new low (downtrend). Ignores what happened inside the range</td></tr>
<tr style="background:rgba(156,39,176,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Statistical</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Regression slope t-stat</td><td style="padding:4px 8px;text-align:left;vertical-align:top">slope of log price on time over the last <i>N</i> months</td><td style="padding:4px 8px;text-align:left;vertical-align:top">p<sub>s</sub> = a + b·s + e<sub>s</sub> over the window; signal = tanh( t(b̂) / 2 )</td><td style="padding:4px 8px;text-align:left;vertical-align:top"><i>N</i></td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: the slope of a straight line fitted through the window's prices, relative to how tightly prices follow that line. A steep slope with little scatter is a strong trend; one extreme point can distort it</td></tr>
<tr style="background:rgba(156,39,176,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Statistical</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Persistence tests</td><td style="padding:4px 8px;text-align:left;vertical-align:top">variance ratio, Hurst exponent, autocorrelation</td><td style="padding:4px 8px;text-align:left;vertical-align:top">VR(q) = Var( r<sub>t</sub><sup>(q)</sup> ) / ( q · Var( r<sub>t</sub> ) ); VR &gt; 1 means trending</td><td style="padding:4px 8px;text-align:left;vertical-align:top">horizon <i>q</i></td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: whether moves tend to continue or reverse. A ratio above 1 means multi-month moves are larger than random-walk scaling would imply, so trend following has had something to exploit. Gives no direction, only whether trends exist</td></tr>
<tr style="background:rgba(156,39,176,0.20)"><td style="padding:4px 8px;text-align:left;vertical-align:top">Statistical</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Changepoint / regime</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Kalman trend, HMM, CUSUM</td><td style="padding:4px 8px;text-align:left;vertical-align:top">local trend: r<sub>t</sub> = μ<sub>t</sub> + ε<sub>t</sub>, μ<sub>t</sub> = μ<sub>t-1</sub> + η<sub>t</sub>; CUSUM: S<sub>t</sub> = max( 0, S<sub>t-1</sub> + r<sub>t</sub> − k )</td><td style="padding:4px 8px;text-align:left;vertical-align:top">model-specific</td><td style="padding:4px 8px;text-align:left;vertical-align:top">Looks at: a hidden trend that is estimated from noisy returns (Kalman, HMM), or a running sum of returns that signals when the average return has shifted (CUSUM). Aims to spot when the trend itself changes</td></tr>
</table>

**Implementable now** (monthly returns only): trailing-return family, MA family via the cumulated return index, breakout, regression t-stat, persistence tests. Daily bars (OHLC, volume, open interest) exist for several of these and could support daily-frequency versions of the MA and breakout measures. Families that need other data or other targets are in 1C.

### 1B. What the measures look like

One synthetic market with known regimes (trend up, range, trend down, a sharp reversal, range), so each measure can be compared with what it should detect. Each panel shows the log price index (grey) with the measure overlaid: lines on the price axis for level-type measures, a score on the right axis for the others. **Background shading is the sign of the forecast** (green long, red short). Synthetic data only: it shows how each measure *behaves* (lag, whipsaws in the range), not which one works in futures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EX_SEED = 7
EX_VOL = 0.03  # monthly return volatility
EX_REGIMES = [("uptrend", 60, 0.02), ("range", 48, 0.0), ("downtrend", 48, -0.02), ("reversal", 36, 0.03), ("range", 48, 0.0)]  # name, months, drift
LOOKBACK, SKIP_MONTHS = 12, 1
BLEND_LOOKBACKS = (1, 3, 12)
FAST, SLOW = 3, 12
BREAKOUT_N, SLOPE_N = 12, 12
VR_Q, VR_WINDOW = 6, 60  # variance-ratio horizon and estimation window
PRICE_COLOR, LINE_COLORS = "0.35", ("tab:blue", "tab:orange")
LONG_COLOR, SHORT_COLOR = "tab:green", "tab:red"


def make_example(regimes, vol, seed):
    rng = np.random.default_rng(seed)
    drift = np.concatenate([np.full(months, mu) for _, months, mu in regimes])
    ret = pd.Series(drift + rng.normal(0, vol, len(drift)), index=pd.period_range("2000-01", periods=len(drift), freq="M"))
    ret.index = ret.index.to_timestamp()
    return ret


ex_ret = make_example(EX_REGIMES, EX_VOL, EX_SEED)
ex_level = np.log1p(ex_ret).cumsum()
ex_bounds = ex_ret.index[np.cumsum([m for _, m, _ in EX_REGIMES])[:-1]]

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(ex_level, color=PRICE_COLOR)
for start, (name, months, _) in zip([ex_ret.index[0], *ex_bounds], EX_REGIMES):
    ax.text(start + pd.DateOffset(months=months // 2), ax.get_ylim()[1], name, ha="center", va="top", fontsize=9)
for b in ex_bounds:
    ax.axvline(b, color="0.7", ls=":")
ax.set_title("Synthetic market: log price index and the regimes that generated it")
plt.show()

In [ ]:
def rolling_ols(level, n):
    # fitted value at the window end and slope t-stat of log price on time
    y = np.lib.stride_tricks.sliding_window_view(level.to_numpy(), n)
    x = np.arange(n) - (n - 1) / 2
    sxx = (x**2).sum()
    slope = (y * x).sum(axis=1) / sxx
    resid = y - y.mean(axis=1, keepdims=True) - slope[:, None] * x
    t = slope / np.sqrt((resid**2).sum(axis=1) / (n - 2) / sxx)
    index = level.index[n - 1 :]
    return pd.Series(y.mean(axis=1) + slope * x[-1], index=index), pd.Series(t, index=index)


def build_example_measures(ret, level):
    ma = level.rolling(LOOKBACK).mean()
    fast, slow = level.rolling(FAST).mean(), level.rolling(SLOW).mean()
    ef, es = level.ewm(span=FAST, min_periods=SLOW).mean(), level.ewm(span=SLOW, min_periods=SLOW).mean()
    box = level.rolling(BREAKOUT_N)
    low, high = box.min(), box.max()
    fitted, slope_t = rolling_ols(level, SLOPE_N)
    roll = ret.rolling(LOOKBACK)
    var_ratio = level.diff(VR_Q).rolling(VR_WINDOW).var() / (VR_Q * ret.rolling(VR_WINDOW).var())
    return {
        "trailing": dict(title=f"Trailing {LOOKBACK}m return", score=level.diff(LOOKBACK)),
        "skip": dict(title=f"Skip-month ({LOOKBACK}-{SKIP_MONTHS})", score=level.shift(SKIP_MONTHS) - level.shift(LOOKBACK)),
        "blend": dict(title=f"Multi-horizon blend {BLEND_LOOKBACKS}", score=sum(np.sign(level.diff(k)) for k in BLEND_LOOKBACKS) / len(BLEND_LOOKBACKS)),
        "tstat": dict(title=f"Return t-stat ({LOOKBACK}m)", score=np.tanh(roll.mean() / (roll.std() / np.sqrt(LOOKBACK)))),
        "ma": dict(title=f"Price vs {LOOKBACK}m MA", lines={"MA": ma}, score=level - ma, on_price=True),
        "cross": dict(title=f"MA crossover {FAST}/{SLOW}", lines={"fast MA": fast, "slow MA": slow}, score=fast - slow, on_price=True),
        "ewmac": dict(title=f"EWMAC {FAST}/{SLOW}", lines={"fast EMA": ef, "slow EMA": es}, score=ef - es, on_price=True),
        "breakout": dict(title=f"Breakout ({BREAKOUT_N}m range)", lines={"high": high, "low": low}, score=(level - low) / (high - low) - 0.5, on_price=True),
        "slope": dict(title=f"Regression slope t-stat ({SLOPE_N}m)", lines={"fitted": fitted}, score=slope_t, on_price=True),
        "persistence": dict(title=f"Variance ratio VR({VR_Q}), {VR_WINDOW}m window", score=var_ratio, ref=1.0, no_shade=True),
    }


def draw_panel(ax, level, spec):
    ax.plot(level, color=PRICE_COLOR, lw=1.2, label="log price")
    for color, (name, line) in zip(LINE_COLORS, spec.get("lines", {}).items()):
        ax.plot(line, color=color, lw=1.3, label=name)
    score = spec["score"]
    if not spec.get("no_shade"):
        for sign, color in ((1, LONG_COLOR), (-1, SHORT_COLOR)):
            ax.fill_between(score.index, 0, 1, where=(np.sign(score) == sign), transform=ax.get_xaxis_transform(), color=color, alpha=0.13, lw=0)
    if not spec.get("on_price"):  # score shares no scale with price: right axis
        twin = ax.twinx()
        twin.plot(score, color="tab:purple", lw=1, label="score")
        twin.axhline(spec.get("ref", 0.0), color="tab:purple", lw=0.6, ls=":")
        twin.tick_params(axis="y", labelcolor="tab:purple", labelsize=7)
    ax.set_title(spec["title"], fontsize=10)
    ax.tick_params(labelsize=7)
    ax.legend(loc="best", fontsize=7, frameon=False)


ex_measures = build_example_measures(ex_ret, ex_level)
cols = 3
rows = -(-len(ex_measures) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, 3.1 * rows), sharex=True)
for ax, spec in zip(axes.flat, ex_measures.values()):
    draw_panel(ax, ex_level, spec)
for ax in list(axes.flat)[len(ex_measures) :]:
    ax.set_visible(False)
fig.suptitle("Trend measures on the synthetic market (shading = sign of forecast; purple = score, right axis)", y=1.0)
fig.tight_layout()
plt.show()

What to read from the panels:

- **Lag.** Every measure turns after the regime changes; the 12-month ones take most of a year to flip after the reversal, the 3/12 crossover flips sooner.
- **Range regimes.** Sign-only measures flip repeatedly while there is no trend (whipsaws); the continuous scores (t-stat, EWMAC, blend) at least change size around zero, which a position-sizing rule can use, but they still cross zero often.
- **Same information, different weighting.** Trailing return, price vs MA and the regression slope read the same window and mostly agree; they differ near turns.
- **Persistence** (variance ratio) gives no direction: VR above 1 says past moves have tended to continue, which is the property a trend follower needs.

### 1C. Families we are exploring but not testing here

These are on the research list and are **not** in the grid, the heatmaps or the overfitting checks below. They need data or a different target that the pipeline does not yet have, so they are described here only.

| Family | Idea | Why try it | Main obstacle | Data needed |
|---|---|---|---|---|
| Fundamental | Trend from drivers: carry (futures curve slope, roll yield), macro, flows, positioning (COT), inventories, value | Different information from price, so it may diversify a price-trend book and can lead price | Market-specific; publication lags; lookahead risk | Term structure, macro, positioning: not in current data (one rolled series per market) |
| Event | A discrete shock (policy, weather, supply, earnings) starts a trend | Gives an economic reason for a trend to start | Hard to time-stamp without lookahead; few observations; not systematic | Dated event data |
| Turning points | Predict where a trend ends: drawdown from peak, trend exhaustion, signal decay | Targets the main weakness of trend following (giving back gains at reversals) | Reversals are rare and noisy; costly when wrong | Prices only (could be tested later), plus a distinct target |

TSMOM and QPS use none of these. Nothing here has been implemented, so none of it counts toward the trial count in section 5.

## 2. Data and conventions

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # project root, so `src` imports

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.alpha_model import CompoundedReturnSignal, SumReturnSignal, lagged
from src.data_cleaning import clean
from src.data_loader import load_asset_map, load_daily_prices, load_monthly_returns

START, END = "1976-01", "2014-12"
INFO_LAG = 1  # a forecast made at the end of month t trades in month t+1
VOL_WINDOW = 36  # trailing months for the per-market volatility scale
MIN_MONTHS = 60  # minimum months of signal history for a market to enter the grid

cleaned = clean(load_monthly_returns(), load_daily_prices(), load_asset_map())
returns = cleaned.returns.copy()
returns.index = returns.index.to_period("M")
returns = returns.loc[START:END]
asset_class = cleaned.asset_map["AssetClass"].reindex(returns.columns)
print(returns.shape, asset_class.value_counts().to_dict())

Conventions that every signal and test below follows:

- A signal is an object with `forecast(returns)` returning a panel of the same shape, **using returns through month t only**, NaN without a full window. This is the existing `Signal` protocol in `src/alpha_model/signals.py`.
- The tradable position is `lagged(forecast, INFO_LAG)`; the lag is applied in one place only (`signal_returns`).
- Strategy returns are position x market return divided by the market's **lagged** trailing volatility, so each market contributes comparable risk. Gross of costs; turnover is reported instead.
- The cumulated log-return index bridges internal return gaps with zero. The window checks use `min_periods`, so a gap inside a window makes the signal NaN for return-based signals; MA, EWMAC and breakout inherit the bridge. Treat their results near gaps with care.

## 3. Signal implementations

In [ ]:
def _log_index(returns):
    return np.log1p(returns).cumsum()


def _trailing_vol(returns, window=VOL_WINDOW):
    return returns.rolling(window, min_periods=window).std()


class SkipMonthSignal:
    """Sign of the return over months t-L+1 .. t-skip (skips the latest `skip` months)."""

    def __init__(self, lookback, skip=1):
        self.lookback, self.skip = lookback, skip

    def forecast(self, returns):
        log_r = np.log1p(returns)
        window = log_r.shift(self.skip).rolling(self.lookback - self.skip, min_periods=self.lookback - self.skip).sum()
        return np.sign(window)


class BlendSignal:
    """Average of compounded-return signs over several lookbacks, in [-1, 1]."""

    def __init__(self, lookbacks=(1, 3, 12)):
        self.parts = [CompoundedReturnSignal(lb) for lb in lookbacks]

    def forecast(self, returns):
        # all-or-nothing: NaN until the longest window is complete
        return sum(p.forecast(returns) for p in self.parts) / len(self.parts)


class TStatSignal:
    """Trailing mean return over its standard error, squashed to (-1, 1) by tanh."""

    def __init__(self, lookback):
        self.lookback = lookback

    def forecast(self, returns):
        roll = returns.rolling(self.lookback, min_periods=self.lookback)
        t = roll.mean() / (roll.std() / np.sqrt(self.lookback))
        return np.tanh(t)


class PriceVsMASignal:
    """Sign of the log index minus its L-month moving average."""

    def __init__(self, lookback):
        self.lookback = lookback

    def forecast(self, returns):
        level = _log_index(returns)
        return np.sign(level - level.rolling(self.lookback, min_periods=self.lookback).mean())


class MACrossSignal:
    """Sign of fast MA minus slow MA of the log index."""

    def __init__(self, fast, slow):
        if fast >= slow:
            raise ValueError("fast window must be shorter than slow window")
        self.fast, self.slow = fast, slow

    def forecast(self, returns):
        level = _log_index(returns)
        fast = level.rolling(self.fast, min_periods=self.fast).mean()
        slow = level.rolling(self.slow, min_periods=self.slow).mean()
        return np.sign(fast - slow)


class EWMACSignal:
    """Fast minus slow exponential MA of the log index, over trailing volatility, tanh-squashed."""

    def __init__(self, fast, slow):
        if fast >= slow:
            raise ValueError("fast span must be shorter than slow span")
        self.fast, self.slow = fast, slow

    def forecast(self, returns):
        level = _log_index(returns)
        diff = level.ewm(span=self.fast, min_periods=self.slow).mean() - level.ewm(span=self.slow, min_periods=self.slow).mean()
        return np.tanh(diff / _trailing_vol(returns))


class BreakoutSignal:
    """Position of the log index in its trailing N-month range, mapped to [-1, 1]."""

    def __init__(self, lookback):
        self.lookback = lookback

    def forecast(self, returns):
        level = _log_index(returns)
        roll = level.rolling(self.lookback, min_periods=self.lookback)
        low, high = roll.min(), roll.max()
        return (2 * (level - low) / (high - low).where(high > low) - 1)


class SlopeTStatSignal:
    """t-statistic of the OLS slope of the log index on time over N months, tanh-squashed."""

    def __init__(self, lookback):
        self.lookback = lookback

    def forecast(self, returns):
        x = np.arange(self.lookback) - (self.lookback - 1) / 2
        sxx = (x**2).sum()

        def t_stat(y):
            if np.isnan(y).any():
                return np.nan
            slope = (x * (y - y.mean())).sum() / sxx
            resid = y - y.mean() - slope * x
            se = np.sqrt((resid**2).sum() / (self.lookback - 2) / sxx)
            return slope / se if se > 0 else np.nan

        level = _log_index(returns)
        return np.tanh(level.rolling(self.lookback, min_periods=self.lookback).apply(t_stat, raw=True) / 2)

Registry of the candidate grid. Every entry is one **trial**: the count matters in section 7. Each entry also carries a **family** (its measure type), so section 4 can keep only the best parameter choice per family. New measures are added here without touching the test code (open-closed).

The grid is wide on purpose: ten lookbacks from 3 to 36 months for the single-window measures, skip lengths, fast/slow pairs with fast <= slow/2, and several horizon blends. Sum and compounded return are near-duplicates (they differ only through compounding) and are kept as separate families because the baseline is the sum.

In [ ]:
LOOKBACKS = (3, 4, 6, 9, 12, 15, 18, 24, 30, 36)  # months, single-window measures
SKIP_LOOKBACKS, SKIPS = (6, 9, 12, 18, 24, 36), (1, 2)
CROSS_FAST, CROSS_SLOW = (2, 3, 4, 6, 9), (6, 9, 12, 18, 24, 36)  # pairs kept when fast <= slow / 2
BLEND_SETS = ((1, 3, 12), (3, 6, 12), (1, 3, 6, 12), (6, 12, 24), (3, 6, 12, 24), (1, 3, 6, 12, 24))


def build_registry():
    """Returns (signals by name, family of each name). One entry is one trial."""
    reg, family = {}, {}

    def add(fam, label, signal):
        name = f"{fam}_{label}"
        reg[name], family[name] = signal, fam

    for lb in LOOKBACKS:
        add("sum", lb, SumReturnSignal(lb))
        add("compound", lb, CompoundedReturnSignal(lb))
        add("tstat", lb, TStatSignal(lb))
        add("price_vs_ma", lb, PriceVsMASignal(lb))
        add("breakout", lb, BreakoutSignal(lb))
        add("slope_t", lb, SlopeTStatSignal(lb))
    for lb in SKIP_LOOKBACKS:
        for skip in SKIPS:
            add("skip", f"{skip}_{lb}", SkipMonthSignal(lb, skip))
    for fast in CROSS_FAST:
        for slow in CROSS_SLOW:
            if 2 * fast <= slow:
                add("ma_cross", f"{fast}_{slow}", MACrossSignal(fast, slow))
                add("ewmac", f"{fast}_{slow}", EWMACSignal(fast, slow))
    for lookbacks in BLEND_SETS:
        add("blend", "_".join(map(str, lookbacks)), BlendSignal(lookbacks))
    return reg, family


REGISTRY, FAMILY = build_registry()
print(len(REGISTRY), "trials in", len(set(FAMILY.values())), "families:", pd.Series(FAMILY).value_counts().to_dict())

## 4. Per-market test scaffold

Every variant is run on every market (hit rate, annualised Sharpe and its t-statistic, rank information coefficient, turnover). The full table is kept for the overfitting checks, which must count **all** trials. What is *reported* is shorter: for each family and market, only the **best parameter choice** (highest Sharpe), so a family appears once per market, not once per lookback.

In [ ]:
def signal_returns(signal, returns, lag=INFO_LAG):
    """Per-market vol-scaled strategy returns. The only place the information lag is applied."""
    position = lagged(signal.forecast(returns), lag)
    scale = lagged(_trailing_vol(returns), 1)  # volatility known before the month starts
    return position * returns / scale, position


def ann(series):
    """Annualised Sharpe of a monthly return series."""
    return series.mean() / series.std() * np.sqrt(12)


def _market_stats(strat, position, returns):
    both = strat.dropna()
    n = len(both)
    if n < MIN_MONTHS:
        return None
    pair = pd.concat([position, returns], axis=1).dropna()
    return {
        "months": n,
        "sharpe": ann(both),
        "t_stat": both.mean() / (both.std() / np.sqrt(n)),
        "hit_rate": (both[both != 0] > 0).mean(),
        "ic_spearman": pair.corr(method="spearman").iloc[0, 1],
        "turnover": position.diff().abs().mean(),
    }


def evaluate_grid(registry, returns, lag=INFO_LAG):
    rows, pooled, panel = [], {}, {}
    for name, signal in registry.items():
        strat, position = signal_returns(signal, returns, lag)
        panel[name] = strat
        pooled[name] = strat.mean(axis=1)  # equal-weight across markets with a position
        for market in returns.columns:
            stats_row = _market_stats(strat[market], position[market], returns[market])
            if stats_row:
                rows.append({"family": FAMILY[name], "signal": name, "market": market, "asset_class": asset_class[market], **stats_row})
    return pd.DataFrame(rows), pd.DataFrame(pooled), panel


def best_per_market(grid):
    """One row per (family, market): the variant with the highest in-sample Sharpe."""
    return grid.loc[grid.groupby(["family", "market"]).sharpe.idxmax()].reset_index(drop=True)


grid, pooled, panel = evaluate_grid(REGISTRY, returns)
best_cells = best_per_market(grid)
print(grid.shape, "variant-market cells ->", best_cells.shape, "best-per-family cells;", pooled.shape[1], "pooled series")

In [ ]:
# Best-parameter Sharpe: families down, markets across, ordered by asset class
heat = best_cells.pivot(index="family", columns="market", values="sharpe")
order = asset_class.sort_values().index.intersection(heat.columns)
fig, ax = plt.subplots(figsize=(16, 4.5), layout="constrained")
im = ax.imshow(heat[order], aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat.index)), heat.index, fontsize=8)
ax.set_title("Best-parameter Sharpe by family and market (in-sample, gross, vol-scaled, lagged one month)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

summary = best_cells.groupby("family").agg(
    markets=("market", "size"),
    median_best_sharpe=("sharpe", "median"),
    share_positive=("sharpe", lambda s: (s > 0).mean()),
    median_ic=("ic_spearman", "median"),
    turnover=("turnover", "median"),
    distinct_choices=("signal", "nunique"),
    most_common_choice=("signal", lambda s: s.mode().iloc[0]),
)
summary.sort_values("median_best_sharpe", ascending=False).round(3)

In [ ]:
# Which parameter wins most often in each asset class
best_cells.pivot_table(index="family", columns="asset_class", values="signal", aggfunc=lambda s: s.mode().iloc[0])

**Reading the table honestly.** Each cell is the *maximum* over that family's parameters, chosen on the same data it is scored on, so it overstates what a fixed choice would earn. `distinct_choices` shows how unstable the winner is across markets: if nearly every market picks a different parameter, the choice is mostly noise. Section 7 prices the selection.

**Best-parameter Sharpe by family and asset class.** Median across the markets in each class of the family's best-parameter Sharpe (in-sample, gross). The last row counts the markets in the class and the last column is all markets.

In [ ]:
by_class = best_cells.pivot_table(index="family", columns="asset_class", values="sharpe", aggfunc="median")
by_class["ALL"] = best_cells.groupby("family").sharpe.median()
by_class = by_class.loc[by_class["ALL"].sort_values(ascending=False).index]
counts = best_cells.drop_duplicates("market").groupby("asset_class").size()
by_class.loc["markets"] = [*counts.reindex(by_class.columns[:-1]), counts.sum()]
by_class.round(2)

## 5. Best signal for each asset class as a whole

Section 4 picked the best parameter for each *security*, which is noisy: one market's history is short and the winner moves with the sample. Here the same grid is scored on the **asset class as a unit**: for each variant, the strategy returns of the markets in a class are equal-weighted into one class portfolio, and the variant with the highest class-portfolio Sharpe wins. Pooling 7 to 26 markets gives a steadier estimate, and it answers a design question directly: should the signal differ by asset class?

Two cautions. The winner is still the maximum over the whole grid chosen on the data it is scored on, so it overstates what a fixed choice would earn: the table shows where the baseline `sum_12` ranks among all variants, and section 7 prices the selection. Classes differ in size, so a small class (FX, equities) gives a noisier ranking than a large one.

In [ ]:
BASELINE = "sum_12"
TOP_N = 3  # variants listed per asset class


def class_grid(panel, grid):
    """Sharpe, t-stat and turnover of every variant on each asset class's equal-weight portfolio."""
    rows = []
    for name, strat in panel.items():
        for cls, markets in asset_class.groupby(asset_class).groups.items():
            ret = strat[list(markets)].mean(axis=1).dropna()
            if len(ret) >= MIN_MONTHS:
                rows.append({"family": FAMILY[name], "signal": name, "asset_class": cls, "months": len(ret), "sharpe": ann(ret), "t_stat": ret.mean() / (ret.std() / np.sqrt(len(ret)))})
    turnover = grid.groupby(["signal", "asset_class"]).turnover.mean().rename("turnover").reset_index()
    return pd.DataFrame(rows).merge(turnover, on=["signal", "asset_class"])


class_results = class_grid(panel, grid)
class_best = class_results.loc[class_results.groupby(["family", "asset_class"]).sharpe.idxmax()]
print(class_results.shape, "variant-class cells ->", class_best.shape, "best-per-family cells")

In [ ]:
# Best-parameter Sharpe of the class portfolio, family by asset class (in-sample, gross)
class_table = class_best.pivot(index="family", columns="asset_class", values="sharpe")
class_table = class_table.loc[class_table.mean(axis=1).sort_values(ascending=False).index]
class_table.loc["baseline " + BASELINE] = class_results[class_results.signal == BASELINE].set_index("asset_class").sharpe
class_table.round(2)

In [ ]:
# The winning parameter for each family and class
class_best.pivot(index="family", columns="asset_class", values="signal").loc[class_table.index[:-1]]

In [ ]:
# Top variants per class, and where the baseline ranks among all of them
n_trials = class_results.signal.nunique()
ranked = class_results.assign(rank=class_results.groupby("asset_class").sharpe.rank(ascending=False).astype(int))
rows = {}
for cls, g in ranked.groupby("asset_class"):
    g = g.sort_values("sharpe", ascending=False)
    base = g[g.signal == BASELINE].iloc[0]
    rows[cls] = {
        **{f"top {i}": f"{r.signal} ({r.sharpe:.2f})" for i, r in enumerate(g.head(TOP_N).itertuples(), 1)},
        "baseline": f"{base.sharpe:.2f}",
        "baseline rank": f"{base['rank']} of {n_trials}",
        "best minus baseline": round(g.sharpe.iloc[0] - base.sharpe, 2),
    }
pd.DataFrame(rows).T

## 6. Does combining signals help?

Different measures read the same price history, so they may agree most of the time and add little. The test: average the forecasts of several signals (equal weights, each forecast already lies in [-1, 1]) and compare the result with its components and with the TSMOM baseline `sum_12`.

- **Combinations are fixed in advance**, not picked from the grid by performance: picking the best combination would repeat the selection problem of section 7. Each combination still counts as a new trial.
- **Same months for everything**: all comparisons use the months in which every one of the 126 signals is available, so a long-lookback signal does not get a different sample.
- **Diversification gain** = combined Sharpe minus the average Sharpe of its components. It is large only if the components are not highly correlated (`avg_corr`).
- **Versus baseline** = combined pooled Sharpe minus `sum_12`, with a 90% block-bootstrap interval (12-month blocks). If the interval contains zero, the data cannot tell the combination from the baseline.

In [ ]:
class CombinedSignal:
    """Equal-weight average of component forecasts, each in [-1, 1]; NaN until every component has a forecast."""

    def __init__(self, signals):
        self.signals = list(signals)

    def forecast(self, returns):
        return sum(s.forecast(returns) for s in self.signals) / len(self.signals)


BOOT_DRAWS, BOOT_BLOCK, BOOT_SEED, BOOT_LEVEL = 2000, 12, 0, 0.90
COMBOS = {
    "trailing return @12m": ["sum_12", "skip_1_12", "tstat_12"],
    "moving average @12m": ["price_vs_ma_12", "ma_cross_3_12", "ewmac_3_12"],
    "range + statistical @12m": ["breakout_12", "slope_t_12"],
    "all families @12m": ["sum_12", "skip_1_12", "tstat_12", "price_vs_ma_12", "ma_cross_3_12", "ewmac_3_12", "breakout_12", "slope_t_12", "blend_1_3_12"],
    "sum, all lookbacks": [f"sum_{lb}" for lb in LOOKBACKS],
    "breakout, all lookbacks": [f"breakout_{lb}" for lb in LOOKBACKS],
    "all trials": list(REGISTRY),
}
missing = {n for names in COMBOS.values() for n in names} - set(REGISTRY)
assert not missing, f"unknown signals in COMBOS: {missing}"
window = pooled.dropna().index  # months where every signal has a forecast


def combo_returns(names):
    strat, position = signal_returns(CombinedSignal([REGISTRY[n] for n in names]), returns)
    return strat.loc[window], position.loc[window]


def sharpe_gain_interval(a, b):
    """Annualised Sharpe(a) - Sharpe(b) and its block-bootstrap interval, resampling months jointly."""
    both = pd.concat([a, b], axis=1).dropna().to_numpy()
    n = len(both)
    rng = np.random.default_rng(BOOT_SEED)
    starts = rng.integers(0, n, size=(BOOT_DRAWS, -(-n // BOOT_BLOCK)))
    idx = ((starts[:, :, None] + np.arange(BOOT_BLOCK)) % n).reshape(BOOT_DRAWS, -1)[:, :n]
    sharpe = lambda x: x.mean(axis=1) / x.std(axis=1, ddof=1) * np.sqrt(12)
    gain = sharpe(both[idx, 0]) - sharpe(both[idx, 1])
    lo, hi = np.quantile(gain, [(1 - BOOT_LEVEL) / 2, (1 + BOOT_LEVEL) / 2])
    return ann(pd.Series(both[:, 0])) - ann(pd.Series(both[:, 1])), lo, hi


def combo_row(names, base_pooled):
    strat, position = combo_returns(names)
    combo_pooled = strat.mean(axis=1)
    market_sharpes = [s["sharpe"] for m in strat if (s := _market_stats(strat[m], position[m], returns[m].loc[window]))]
    parts = pooled.loc[window, names]
    corr = parts.corr().to_numpy()
    gain, lo, hi = sharpe_gain_interval(combo_pooled, base_pooled)
    return {
        "components": len(names),
        "pooled_sharpe": ann(combo_pooled),
        "avg_component_sharpe": parts.apply(ann).mean(),
        "avg_corr": corr[np.triu_indices_from(corr, k=1)].mean() if len(names) > 1 else np.nan,
        "median_market_sharpe": np.median(market_sharpes),
        "turnover": position.diff().abs().mean().mean(),
        "vs_baseline": gain,
        f"vs_baseline_{BOOT_LEVEL:.0%}_low": lo,
        f"vs_baseline_{BOOT_LEVEL:.0%}_high": hi,
    }


base_pooled = combo_returns([BASELINE])[0].mean(axis=1)
combo_table = pd.DataFrame({"baseline " + BASELINE: combo_row([BASELINE], base_pooled), **{k: combo_row(v, base_pooled) for k, v in COMBOS.items()}}).T
print(f"{len(window)} common months, {window[0]} to {window[-1]}")
combo_table.round(2)

## 7. Overfitting checks

Five checks. The first four map to challenges that bite the grid directly; the fifth is a validation framework that tests the whole procedure "choose the best parameter per market".

| # | Challenge | Why it bites here | Check below |
|---|---|---|---|
| 1 | Multiple testing | many variants x ~55 markets; the best cells look significant by chance | Holm / Benjamini-Hochberg over **all** trials, deflated Sharpe |
| 2 | Parameter snooping | picking the best lookback in-sample then reporting the same sample | walk-forward selection |
| 3 | Fragile optimum | a spike in the Sharpe surface is noise; a plateau is signal | lookback sensitivity |
| 4 | Costs and turnover | fast signals look best gross | gross vs net Sharpe, break-even cost |
| 5 | Validation framework | one train/test split is one draw; selection risk stays hidden | purged and embargoed combinatorial cross-validation (CPCV) |

### 7.1 Multiple testing and deflated Sharpe

p-values from the per-cell t-statistics (normal approximation) are adjusted for the **whole** grid, not only for the best-per-family cells that section 4 shows: the selection step is itself a test. The deflated Sharpe asks whether the best pooled Sharpe still exceeds the maximum expected from `N` zero-skill trials. The independent-trial count `N` is unknown (trials are correlated), so it is reported for a range.

In [ ]:
def adjust_pvalues(p, method="holm"):
    p = np.asarray(p)
    order = np.argsort(p)
    m = len(p)
    ranked = p[order]
    if method == "holm":
        adj = np.maximum.accumulate((m - np.arange(m)) * ranked)
    elif method == "bh":
        adj = np.minimum.accumulate((m / np.arange(1, m + 1) * ranked)[::-1])[::-1]
    else:
        raise ValueError(method)
    out = np.empty(m)
    out[order] = np.minimum(adj, 1)
    return out


def expected_max_sharpe(n_trials, sharpe_std):
    """Expected maximum of n zero-mean Sharpe estimates (Bailey & Lopez de Prado)."""
    gamma = 0.5772156649
    z = stats.norm.ppf
    return sharpe_std * ((1 - gamma) * z(1 - 1 / n_trials) + gamma * z(1 - 1 / (n_trials * np.e)))


def deflated_sharpe(returns_series, n_trials, sharpe_std):
    """Probability that the true Sharpe exceeds the expected maximum under n_trials of luck (monthly units)."""
    r = returns_series.dropna()
    sr = r.mean() / r.std()
    benchmark = expected_max_sharpe(n_trials, sharpe_std)
    skew, kurt = stats.skew(r), stats.kurtosis(r, fisher=False)
    denom = np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr**2)
    return stats.norm.cdf((sr - benchmark) * np.sqrt(len(r) - 1) / denom)


grid["p_value"] = 2 * stats.norm.sf(grid.t_stat.abs())
grid["p_holm"] = adjust_pvalues(grid.p_value, "holm")
grid["p_bh"] = adjust_pvalues(grid.p_value, "bh")
best_cells = best_per_market(grid)
print(f"all {len(grid)} trials | raw p<0.05: {(grid.p_value < .05).sum()} | Holm<0.05: {(grid.p_holm < .05).sum()} | BH<0.05: {(grid.p_bh < .05).sum()}")
print(f"cells expected significant by chance alone at 5%: {0.05 * len(grid):.0f}")
print(f"best-per-family cells ({len(best_cells)}), adjusted over all {len(grid)} trials | raw p<0.05: {(best_cells.p_value < .05).sum()} | Holm<0.05: {(best_cells.p_holm < .05).sum()} | BH<0.05: {(best_cells.p_bh < .05).sum()}")

monthly_sharpe = pooled.apply(lambda s: s.mean() / s.std())
best = monthly_sharpe.idxmax()
for n_trials in (5, 10, len(REGISTRY)):
    dsr = deflated_sharpe(pooled[best], n_trials, monthly_sharpe.std())
    print(f"best pooled signal {best}: deflated Sharpe probability with N={n_trials}: {dsr:.3f}")

### 7.2 Walk-forward selection

Each December, pick the signal with the best pooled Sharpe over all prior data (minimum 10 years), hold it for the next year, and chain the out-of-sample years. The gap between the in-sample best and this series is the cost of choosing.

In [ ]:
def walk_forward(pooled, min_years=10):
    years = sorted(set(pooled.index.year))
    picks, oos = {}, []
    for year in years[min_years:]:
        history = pooled.loc[: f"{year - 1}-12"]
        sharpe = history.mean() / history.std()
        choice = sharpe.idxmax()
        picks[year] = choice
        oos.append(pooled.loc[str(year), choice])
    return pd.concat(oos), pd.Series(picks)


oos, picks = walk_forward(pooled.dropna(how="all"))
in_sample = pooled.loc[oos.index[0] :]
print(f"walk-forward OOS Sharpe: {ann(oos):.2f} | best fixed signal over same years (hindsight): {in_sample.apply(ann).max():.2f} | median signal: {in_sample.apply(ann).median():.2f}")
print("distinct signals chosen:", picks.nunique(), "| switches:", int((picks != picks.shift()).sum() - 1))

### 7.3 Lookback sensitivity: plateau or spike

In [ ]:
families = {"sum": "sum_", "t-stat": "tstat_", "price vs MA": "price_vs_ma_", "breakout": "breakout_", "slope t-stat": "slope_t_"}
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
for label, prefix in families.items():
    cols = [c for c in pooled.columns if c.startswith(prefix)]
    ax.plot([int(c.removeprefix(prefix)) for c in cols], [ann(pooled[c]) for c in cols], marker="o", label=label)
ax.set(xlabel="Lookback (months)", ylabel="Pooled Sharpe", title="A trustworthy signal has a plateau, not a spike")
ax.axhline(0, color="grey", linewidth=0.6)
ax.legend()
plt.show()

### 7.4 Costs and turnover

Fast signals trade more, so a gross Sharpe flatters them. For the best pooled variant of each family this shows turnover, gross and net Sharpe at an **assumed** cost per unit of notional traded, and the **break-even cost** that drives the net return to zero. The assumed cost is a placeholder; the cost-model stage supplies per-market costs. Costs are charged on the change in notional (position divided by the lagged volatility scale), the same units as the strategy return.

In [ ]:
COST_BPS = 5  # assumed cost per unit of notional traded, in basis points (placeholder, not the cost model)


def cost_profile(signal, returns, cost_bps=COST_BPS):
    strat, position = signal_returns(signal, returns)
    traded = (position / lagged(_trailing_vol(returns), 1)).diff().abs()  # notional traded, same units as strat
    gross, drag = strat.mean(axis=1), traded.mean(axis=1)
    net = gross - cost_bps / 1e4 * drag
    return {
        "turnover": position.diff().abs().mean().mean(),
        "gross_sharpe": ann(gross),
        "net_sharpe": ann(net),
        "breakeven_bps": gross.mean() / drag.mean() * 1e4,
    }


pooled_sharpe = pooled.apply(ann)
best_by_family = pooled_sharpe.groupby(pooled_sharpe.index.map(FAMILY)).idxmax()
pd.DataFrame({fam: {"variant": name, **cost_profile(REGISTRY[name], returns)} for fam, name in best_by_family.items()}).T.sort_values("net_sharpe", ascending=False).round(3)

### 7.5 Purged and embargoed combinatorial cross-validation (CPCV)

A single walk-forward run is one path through history, and one train/test split is one draw. CPCV (Lopez de Prado) tests the *procedure* many ways at once:

1. Split the months into `N` contiguous groups and choose every combination of `k` groups as the test set: `C(N, k)` splits. Each group is tested in `C(N-1, k-1)` splits, so the out-of-sample pieces can be stitched into `C(N-1, k-1)` complete **paths**, each a full out-of-sample history made of different splits.
2. **Purge**: drop training months just before each test block, because a month's strategy return is decided one month earlier (information lag) and would otherwise straddle the boundary.
3. **Embargo**: drop training months just after each test block, because their signals are built from returns inside the test block.
4. In each split, choose the best parameter **per market** using training months only, then score that choice on the test months. This is the section 4 procedure, evaluated out of sample.

Outputs per family: in-sample Sharpe of the chosen parameter, its out-of-sample Sharpe, the **probability of backtest overfitting (PBO)**: the share of choices whose out-of-sample rank is below the median of that family's parameters, and the Sharpe of each stitched path (equal-weight across markets). The reference line is the fixed 12-month sum signal with no selection.

In [ ]:
from itertools import combinations
import warnings

CPCV_GROUPS, CPCV_TEST_GROUPS = 10, 2
PURGE_MONTHS = INFO_LAG + 1  # months dropped before each test block
EMBARGO_MONTHS = 12  # months dropped after each test block
CPCV_MIN_TRAIN, CPCV_MIN_TEST = 36, 12  # valid months needed to score a market


def cpcv_groups(n_obs, n_groups):
    edges = np.linspace(0, n_obs, n_groups + 1).astype(int)
    return [np.arange(a, b) for a, b in zip(edges[:-1], edges[1:])]


def cpcv_splits(groups, n_obs, n_test, purge, embargo):
    """Yield (train_mask, test_mask, test_group_ids), training months purged and embargoed around every test block."""
    for chosen in combinations(range(len(groups)), n_test):
        test = np.zeros(n_obs, dtype=bool)
        train = np.ones(n_obs, dtype=bool)
        for g in chosen:
            lo, hi = groups[g][0], groups[g][-1] + 1
            test[lo:hi] = True
            train[max(0, lo - purge) : hi + embargo] = False
        train &= ~test
        yield train, test, chosen


def masked_sharpe(x, mask, min_obs):
    """Annualised Sharpe per column of x (months, markets) over the masked months; NaN below min_obs valid months."""
    sub = x[mask]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sharpe = np.nanmean(sub, axis=0) / np.nanstd(sub, axis=0, ddof=1) * np.sqrt(12)
    return np.where((~np.isnan(sub)).sum(axis=0) >= min_obs, sharpe, np.nan)


def cpcv_family(cube):
    """cube: (variants, months, markets) strategy returns of one family. Selection per market uses training months only."""
    n_var, n_obs, n_mkt = cube.shape
    cols = np.arange(n_mkt)
    groups = cpcv_groups(n_obs, CPCV_GROUPS)
    chosen_per_split, pooled_oos, is_sr, oos_sr, below_median = [], [], [], [], []
    for train, test, chosen in cpcv_splits(groups, n_obs, CPCV_TEST_GROUPS, PURGE_MONTHS, EMBARGO_MONTHS):
        train_sr = np.stack([masked_sharpe(cube[v], train, CPCV_MIN_TRAIN) for v in range(n_var)])
        test_sr = np.stack([masked_sharpe(cube[v], test, CPCV_MIN_TEST) for v in range(n_var)])
        pick = np.argmax(np.where(np.isnan(train_sr), -np.inf, train_sr), axis=0)
        ok = ~np.isnan(train_sr[pick, cols]) & ~np.isnan(test_sr[pick, cols])
        picked_test = test_sr[pick, cols]
        frac_below = (test_sr < picked_test).sum(axis=0) / np.maximum((~np.isnan(test_sr)).sum(axis=0) - 1, 1)
        is_sr.append(train_sr[pick, cols][ok])
        oos_sr.append(picked_test[ok])
        below_median.append(frac_below[ok] < 0.5)
        selected = np.stack([cube[pick[m], :, m] for m in cols], axis=1)
        selected[:, ~ok] = np.nan
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pooled_oos.append(np.nanmean(selected, axis=1))
        chosen_per_split.append(chosen)
    appearances = {g: [i for i, c in enumerate(chosen_per_split) if g in c] for g in range(len(groups))}
    n_paths = len(appearances[0])
    paths = [np.concatenate([pooled_oos[appearances[g][j]][groups[g]] for g in range(len(groups))]) for j in range(n_paths)]
    path_sharpes = np.array([ann(pd.Series(p).dropna()) for p in paths])
    return {
        "is_sharpe": np.median(np.concatenate(is_sr)),
        "oos_sharpe": np.median(np.concatenate(oos_sr)),
        "pbo": np.concatenate(below_median).mean(),
        "path_sharpes": path_sharpes,
    }


names_by_family = {fam: [n for n, f in FAMILY.items() if f == fam] for fam in dict.fromkeys(FAMILY.values())}
cpcv = {fam: cpcv_family(np.stack([panel[n].to_numpy() for n in names])) for fam, names in names_by_family.items()}
cpcv_table = pd.DataFrame(
    {
        fam: {
            "variants": len(names_by_family[fam]),
            "IS Sharpe (chosen)": r["is_sharpe"],
            "OOS Sharpe (chosen)": r["oos_sharpe"],
            "PBO": r["pbo"],
            "path Sharpe median": np.median(r["path_sharpes"]),
            "path Sharpe min": r["path_sharpes"].min(),
            "path Sharpe max": r["path_sharpes"].max(),
        }
        for fam, r in cpcv.items()
    }
).T.sort_values("path Sharpe median", ascending=False)
n_paths = len(next(iter(cpcv.values()))["path_sharpes"])
print(f"{CPCV_GROUPS} groups, {CPCV_TEST_GROUPS} test groups per split -> {len(list(combinations(range(CPCV_GROUPS), CPCV_TEST_GROUPS)))} splits, {n_paths} paths; purge {PURGE_MONTHS}m, embargo {EMBARGO_MONTHS}m")
cpcv_table.round(3)

In [ ]:
# Path Sharpes per family against the fixed 12-month sum signal (no selection, full sample)
fixed = ann(pooled["sum_12"].dropna())
order = cpcv_table.index
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
for i, fam in enumerate(order):
    ax.scatter([i] * n_paths, cpcv[fam]["path_sharpes"], color="tab:blue", alpha=0.6, s=22)
ax.axhline(fixed, color="tab:red", ls="--", lw=1, label=f"fixed sum_12, no selection ({fixed:.2f})")
ax.axhline(0, color="grey", lw=0.6)
ax.set_xticks(range(len(order)), order, rotation=45, ha="right")
ax.set(ylabel="Out-of-sample path Sharpe", title=f"CPCV: {n_paths} stitched out-of-sample paths per family (per-market parameter choice)")
ax.legend()
plt.show()

## 8. Before any signal is carried forward

1. Fix the candidate list and trial count **before** looking at results; log every variant tried, including discarded ones.
2. Require the signal to survive: multiple-testing adjustment at the true trial count, walk-forward selection, a plateau in the lookback surface, and a CPCV PBO well below 50% with positive path Sharpes.
3. Test it by asset class and by market only as a diagnostic of where the effect lives, not to choose markets.
4. Re-run net of the transaction cost model, comparing against the TREND baseline at equal risk and gross exposure.
5. Freeze the rules, then evaluate once on post-2014 data or paper trading. Do not tune on that data.
6. Graduate the surviving classes from this notebook into `src/alpha_model/signals.py` with tests next to the existing ones.